<a href="https://colab.research.google.com/github/Croop-weed/DeepLearning-parctice-/blob/main/smartbin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
harshitdhaundiyal_dustbin_path = kagglehub.dataset_download('harshitdhaundiyal/dustbin')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/harshitdhaundiyal/dustbin/")

splits = ["train", "valid", "test"]

for split in splits:
    images = list((ROOT / split / "images").glob("*"))
    labels = list((ROOT / split / "labels").glob("*.txt"))

    print(f"\n{split.upper()}")
    print(f"Images : {len(images)}")
    print(f"Labels : {len(labels)}")

In [ ]:
from collections import Counter

counter = Counter()

for split in splits:
    label_dir = ROOT / split / "labels"

    for file in label_dir.glob("*.txt"):

        with open(file) as f:
            for line in f:

                cls = int(line.split()[0])
                counter[cls] += 1

print(counter)

In [ ]:
for split in splits:

    image_dir = ROOT / split / "images"
    label_dir = ROOT / split / "labels"

    unlabeled = []

    for image in image_dir.iterdir():

        txt = label_dir / f"{image.stem}.txt"

        if not txt.exists():
            unlabeled.append(image)

    print(split, len(unlabeled))

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import random

ROOT = Path("/kaggle/input/datasets/harshitdhaundiyal/dustbin")

CLASS_NAMES = {
    0: "bin",
    1: "full"
}

image_dir = ROOT / "train" / "images"
label_dir = ROOT / "train" / "labels"

image_path = random.choice(list(image_dir.glob("*")))

label_path = label_dir / f"{image_path.stem}.txt"

img = cv2.imread(str(image_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

h, w = img.shape[:2]

with open(label_path) as f:
    for line in f:

        cls, xc, yc, bw, bh = map(float, line.split())

        cls = int(cls)

        xmin = int((xc - bw/2) * w)
        ymin = int((yc - bh/2) * h)
        xmax = int((xc + bw/2) * w)
        ymax = int((yc + bh/2) * h)

        cv2.rectangle(
            img,
            (xmin, ymin),
            (xmax, ymax),
            (0,255,0),
            2
        )

        cv2.putText(
            img,
            CLASS_NAMES[cls],
            (xmin, ymin-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,0,0),
            2
        )

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")

In [ ]:
from collections import Counter

combo_counter = Counter()

for split in ["train", "valid", "test"]:

    label_dir = ROOT / split / "labels"

    for file in label_dir.glob("*.txt"):

        classes = set()

        with open(file) as f:
            for line in f:
                classes.add(int(line.split()[0]))

        combo_counter[tuple(sorted(classes))] += 1

print(combo_counter)


In [ ]:
!pip install -q ultralytics

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import ultralytics
from ultralytics import YOLO

print(ultralytics.__version__)

In [ ]:
DATASET = Path("/kaggle/input/datasets/harshitdhaundiyal/dustbin")

print(DATASET)
import yaml

with open(DATASET / "data.yaml") as f:
    data = yaml.safe_load(f)

data

In [ ]:
import yaml

yaml_path = DATASET / "data.yaml"

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["train"] = str(DATASET / "train" / "images")
cfg["val"] = str(DATASET / "valid" / "images")
cfg["test"] = str(DATASET / "test" / "images")

new_yaml = "/kaggle/working/data.yaml"

with open(new_yaml, "w") as f:
    yaml.dump(cfg, f)

print(cfg)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

In [ ]:
results = model.train(
    data=yaml_path,

    # Training
    epochs=100,
    imgsz=640,

    # Multi GPU
    device=[0, 1],

    # Batch
    batch=64,

    workers=8,

    optimizer="auto",
    lr0=0.01,

    # Misc
    project="/kaggle/working/runs",
    name="yolov8s_baseline",
    pretrained=True,
    seed=42,
    patience=20,
    cache=True,
    amp=True,
)